In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import json
import warnings

from src.features import clean_column_names, cast_small_string_variables, DataCleaner

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

# Consistent colour palette across all notebooks
COLORS = {
    'European hornet': '#E07B39',   # orange
    'Asian hornet':    '#3A6EA5',   # blue
    'missing':         '#BDBDBD',   # grey
    'accent':          '#2ECC71',   # green
}

DATA_DIR = Path(r'C:\Users\danie\PycharmProjects\nabu-asian-hornet-project-research\data')
print('Setup complete ✓')
print('Data directory:', DATA_DIR.resolve())
NOTEBOOK_DIR = Path(r'C:\Users\danie\PycharmProjects\nabu-asian-hornet-project-research\notebooks')
print("Notebook directory:", NOTEBOOK_DIR.resolve())

Setup complete ✓
Data directory: C:\Users\danie\PycharmProjects\nabu-asian-hornet-project-research\data
Notebook directory: C:\Users\danie\PycharmProjects\nabu-asian-hornet-project-research\notebooks


In [2]:
df_asian = pd.read_csv(DATA_DIR / 'asian_hornet_DE.csv')
df_european = pd.read_csv(DATA_DIR / 'european_hornet_DE.csv')

print(df_asian.columns.tolist())
print(df_european.columns.tolist())

# Identify columns common to both datasets
common_cols = df_asian.columns.intersection(df_european.columns)
print(f"Common columns: {len(common_cols)}")

# Columns exclusive to European hornet dataset
only_european = df_european.columns.difference(df_asian.columns)
print(f"Only in European hornet: {len(only_european)}")
print(only_european.tolist())

# Columns exclusive to Asian hornet dataset
only_asian = df_asian.columns.difference(df_european.columns)
print(f"Only in Asian hornet: {len(only_asian)}")
print(only_asian.tolist())

# Recommended core columns for comparative analysis
core_cols = [
    "species", "decimalLatitude", "decimalLongitude",
    "year", "month", "stateProvince", "basisOfRecord", "eventDate"
]

# Filter both datasets to core columns only
df_asian_core = df_asian[core_cols]
df_european_core = df_european[core_cols]

print(f"\nAsian hornet core shape:    {df_asian_core.shape}")
print(f"European hornet core shape: {df_european_core.shape}")

['key', 'datasetKey', 'publishingOrgKey', 'installationKey', 'hostingOrganizationKey', 'publishingCountry', 'protocol', 'lastCrawled', 'lastParsed', 'crawlId', 'extensions', 'basisOfRecord', 'occurrenceStatus', 'classifications', 'taxonKey', 'kingdomKey', 'phylumKey', 'classKey', 'orderKey', 'familyKey', 'genusKey', 'speciesKey', 'acceptedTaxonKey', 'scientificName', 'scientificNameAuthorship', 'acceptedScientificName', 'kingdom', 'phylum', 'order', 'family', 'genus', 'species', 'genericName', 'specificEpithet', 'infraspecificEpithet', 'taxonRank', 'taxonomicStatus', 'decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters', 'continent', 'gadm', 'year', 'month', 'day', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'issues', 'lastInterpreted', 'license', 'isSequenced', 'identifiers', 'media', 'facts', 'relations', 'isInCluster', 'datasetID', 'recordedBy', 'identifiedBy', 'dnaSequenceID', 'nucleotideSequence', 'geodeticDatum', 'class', 'countryCode', 'recordedByIDs', 'ident

In [3]:
df_asian_core = df_asian[core_cols]
df_european_core = df_european[core_cols]
#df_asian_core.head() # Vespa velutina
df_european_core.head() # Vespa crabro


,species,decimalLatitude,decimalLongitude,year,month,stateProvince,basisOfRecord,eventDate
0,Vespa crabro,50.8610,6.1533,2000,5.0000,NaN,HUMAN_OBSERVATION,2000-05-23T00:00
1,Vespa crabro,50.2012,10.2044,2000,5.0000,NaN,HUMAN_OBSERVATION,2000-05-05T00:00
2,Vespa crabro,52.9206,13.5533,2000,9.0000,NaN,OBSERVATION,2000-09-01T14:56
3,Vespa crabro,53.5943,10.7510,2000,9.0000,NaN,PRESERVED_SPECIMEN,2000-09-25
4,Vespa crabro,53.4798,10.6664,2000,9.0000,NaN,PRESERVED_SPECIMEN,2000-09-23


In [4]:
# print dtypes for each dataset
df_asian_core.dtypes
df_european_core.dtypes

# cast month and year to integer and empty values to NA
df_asian_core["month"] = pd.to_numeric(df_asian_core["month"], errors="coerce")
df_asian_core["year"] = pd.to_numeric(df_asian_core["year"], errors="coerce")
df_european_core["month"] = pd.to_numeric(df_european_core["month"], errors="coerce")
df_european_core["year"] = pd.to_numeric(df_european_core["year"], errors="coerce")
print(df_asian_core.dtypes)
print(df_european_core.dtypes)
#df_asian_core["stateProvince"].value_counts()
#df_european_core["stateProvince"].value_counts()

combined_data = pd.concat([df_asian_core, df_european_core])


# convert all column names to snake_case
combined_data = clean_column_names(combined_data)

# check for string variables with less than 5 unique values. If less than 5, cast as an object.
cast_small_string_variables(combined_data)
print(combined_data.dtypes)

# we know that there are no records of the asian hornet before 2013. Older data is not helpful for seeing how/when they spread.

# filter combined data to year >= 2010
combined_data = combined_data[combined_data["year"] >= 2010]
combined_data.shape


species                 str
decimalLatitude     float64
decimalLongitude    float64
year                  int64
month                 int64
stateProvince           str
basisOfRecord           str
eventDate               str
dtype: object
species                 str
decimalLatitude     float64
decimalLongitude    float64
year                  int64
month               float64
stateProvince           str
basisOfRecord           str
eventDate               str
dtype: object
species             category
decimallatitude      float64
decimallongitude     float64
year                   int64
month                float64
stateprovince            str
basisofrecord            str
eventdate                str
dtype: object


(78513, 8)

In [7]:
df_asian_clean = (
    DataCleaner(df_asian, core_cols=core_cols)
    .select_core_cols()
    .clean_column_names()
    .drop_null_columns()
    .drop_single_value_columns()
    .remove_special_characters()
    .cast_small_string_variables()
    .get()
)

df_european_clean = (
    DataCleaner(df_european, core_cols=core_cols)
    .select_core_cols()
    .clean_column_names()
    .drop_null_columns()
    .drop_single_value_columns()
    .remove_special_characters()
    .cast_small_string_variables()
    .get()
)